In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib
import os

# Define the absolute path for data and model storage
DATA_DIR = '/home/student/cyberai-lab/data/alerts'
MODEL_DIR = '/home/student/cyberai-lab/models'

# Ensure directories exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# --- Configuration ---
# In a real SOAR, this data would be ingested from SIEM/alerting systems.
# For this example, we'll simulate it.
ALERT_DATA_FILE = os.path.join(DATA_DIR, 'simulated_alerts.csv')
MODEL_FILE = os.path.join(MODEL_DIR, 'alert_prioritizer_rf.joblib')

# --- Data Simulation (Replace with actual data ingestion) ---
def simulate_alert_data(filepath):
    if not os.path.exists(filepath):
        print(f"Simulating alert data and saving to {filepath}...")
        data = {
            'source_ip': ['192.168.1.10', '10.0.0.5', '203.0.113.10', '192.168.1.10', '10.0.0.5', '203.0.113.25', '192.168.1.15', '10.0.0.6', '203.0.113.10', '192.168.1.10'],
            'alert_type': ['BruteForce', 'Malware', 'PortScan', 'BruteForce', 'Malware', 'PortScan', 'BruteForce', 'Malware', 'PortScan', 'BruteForce'],
            'failed_logins': [5, 0, 10, 50, 0, 5, 15, 0, 8, 60],
            'affected_servers': ['webserver', 'dbserver', 'firewall', 'webserver', 'dbserver', 'firewall', 'webserver', 'dbserver', 'firewall', 'webserver'],
            'is_malicious': [0, 1, 0, 1, 1, 0, 0, 1, 0, 1] # 0: Benign, 1: Malicious
        }
        df = pd.DataFrame(data)
        df.to_csv(filepath, index=False)
        print("Simulation complete.")
    else:
        print(f"Alert data already exists at {filepath}.")

simulate_alert_data(ALERT_DATA_FILE)

# --- Model Training Function ---
def train_alert_prioritizer(data_file, model_save_path):
    try:
        df = pd.read_csv(data_file)
        print(f"Loaded {len(df)} records for training.")

        # Feature Engineering: Convert categorical features to numerical
        df['alert_type_encoded'] = df['alert_type'].astype('category').cat.codes
        df['affected_servers_encoded'] = df['affected_servers'].astype('category').cat.codes
        # Simple IP to numerical conversion (for demonstration)
        df['source_ip_numeric'] = df['source_ip'].apply(lambda ip: sum([int(x) for x in ip.split('.')]))

        # Define features (X) and target (y)
        features = ['source_ip_numeric', 'alert_type_encoded', 'failed_logins', 'affected_servers_encoded']
        target = 'is_malicious'

        X = df[features]
        y = df[target]

        # Split data into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        # Initialize and train the Random Forest Classifier
        model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
        print("Training model...")
        model.fit(X_train, y_train)

        # Evaluate the model
        y_pred = model.predict(X_test)
        print("Model training complete. Evaluation:")
        print(classification_report(y_test, y_pred))

        # Save the trained model
        joblib.dump(model, model_save_path)
        print(f"Trained model saved to {model_save_path}")
        return True

    except FileNotFoundError:
        print(f"Error: Data file not found at {data_file}")
        return False
    except Exception as e:
        print(f"An error occurred during training: {e}")
        return False

# --- Prediction Function (to be used by SOAR playbook) ---
def predict_alert_priority(alert_data, model_path):
    try:
        # Load the trained model
        model = joblib.load(model_path)
        print("Model loaded successfully.")

        # Prepare the input alert data for prediction
        # This requires the same feature engineering as during training
        alert_df = pd.DataFrame([alert_data])
        
        # Apply same encoding/transformations as during training
        # NOTE: In a real SOAR, these mappings (e.g., category codes) should be stored or accessible
        # For simplicity, we re-create them here, assuming consistent data.
        alert_df['alert_type_encoded'] = alert_df['alert_type'].astype('category').cat.codes
        alert_df['affected_servers_encoded'] = alert_df['affected_servers'].astype('category').cat.codes
        alert_df['source_ip_numeric'] = alert_df['source_ip'].apply(lambda ip: sum([int(x) for x in ip.split('.')]))

        features = ['source_ip_numeric', 'alert_type_encoded', 'failed_logins', 'affected_servers_encoded']
        X_predict = alert_df[features]

        # Make prediction
        prediction = model.predict(X_predict)[0]
        probability = model.predict_proba(X_predict)[0]

        priority = "High" if prediction == 1 else "Low"
        confidence = probability[prediction]

        print(f"Prediction: {priority} (Confidence: {confidence:.2f})")
        return {"priority": priority, "confidence": confidence, "prediction": int(prediction)}

    except FileNotFoundError:
        print(f"Error: Model file not found at {model_path}")
        return None
    except KeyError as e:
        print(f"Error: Missing expected feature during prediction: {e}. Ensure feature engineering matches training.")
        return None
    except Exception as e:
        print(f"An error occurred during prediction: {e}")
        return None

# --- Main Execution Logic ---
if __name__ == "__main__":
    # Step 1: Train the model if it doesn't exist
    if not os.path.exists(MODEL_FILE):
        print("Model not found. Training the alert prioritizer...")
        if not train_alert_prioritizer(ALERT_DATA_FILE, MODEL_FILE):
            print("Model training failed. Exiting.")
            exit()
        print("\n---\n")
    else:
        print(f"Model found at {MODEL_FILE}. Skipping training.")
        print("\n---\n")

    # Step 2: Simulate a new alert and predict its priority
    print("Simulating a new alert for prediction...")
    new_alert = {
        'source_ip': '203.0.113.50', # External IP
        'alert_type': 'BruteForce', 
        'failed_logins': 120, 
        'affected_servers': 'admin_panel'
    }
    
    print(f"New alert data: {new_alert}")
    prediction_result = predict_alert_priority(new_alert, MODEL_FILE)

    if prediction_result:
        print(f"\nSOAR Playbook Action Result:")
        print(f"  Alert Priority: {prediction_result['priority']}")
        print(f"  Confidence: {prediction_result['confidence']:.2f}")
        
        # Example of how SOAR would use this result:
        if prediction_result['priority'] == 'High':
            print("  Action: Escalate alert to Tier 2 analyst.")
        else:
            print("  Action: Automatically close alert (low confidence/benign).")
    else:
        print("\nFailed to predict alert priority.")

    print("\nIf you see this message → your lab is working perfectly!")